In [6]:
# imports
import pandas as pd
import openpyxl
import numpy as np
import os
import re
from collections import defaultdict

In [2]:
pd.set_option('display.max_rows', None) 

In [5]:
import sys
print(sys.executable)

/Users/emma/projects/cognoscolab/myvenv/bin/python


In [3]:
path = '/Users/emma/projects/cognoscolab/math_anxiety_LLMs'

## Files reading

In [4]:
# open files
data_2024 = []
folder_path = f'{path}/data/2024_human_students_data'
for filename in os.listdir(folder_path):
    if filename.endswith(".xlsx") and filename.startswith("Task_"):
        file_path = os.path.join(folder_path, filename)
        try:
            personal_data = pd.read_excel(file_path, sheet_name="Dati_Demografici", header=None)
        except ValueError:
            personal_data = pd.read_excel(file_path, sheet_name="Dati Demografici", header=None)

        # create a list with the necessary data from each file
        data_2024.append({
            "participant_id": f"student_exp1_{len(data_2024)+1:03}",
            "filename": filename,
            "personal_data": personal_data,
            "questionnaries": pd.read_excel(file_path, sheet_name="Questionari", header=None),
            "associations": pd.read_excel(file_path, sheet_name="Task_1parte_Associazioni", header=None),
            "valence": pd.read_excel(file_path, sheet_name="Task_2parte_Valenza", header=None),
        })

# get all the data into a dataframe with raw data
df_raw_data_2024 = pd.DataFrame(data_2024)

## Personal data extraction

In [3]:
# create a new df to store the clean data
df_data_2024 = pd.DataFrame()
# participant id and filename
df_data_2024['participant_id'] = df_raw_data_2024['participant_id']
df_data_2024['filename'] = df_raw_data_2024['filename']

# number of participants
n_participants = len(df_data_2024)

In [4]:
# get the gender of each participant
genders = []
for i in range(n_participants):
    gender = df_raw_data_2024['personal_data'][i].iloc[8, 1]
    if pd.isna(gender):
        gender = df_raw_data_2024['personal_data'][i].iloc[7, 1]
    genders.append(gender)

# insert the gender in the df with the participant_id, filename
df_data_2024['gender'] = pd.DataFrame(genders)

## MAS-IT data extraction

In [5]:
# create an array to store the answers to the MAS-IT questionnaire (with the number of particpants and 23 questions)
mas_it_array = np.empty((n_participants, 23))

# loop through each participant and extract the answers to the MAS-IT questionnaire
for i in range(n_participants):
    df_questionnaire = df_raw_data_2024['questionnaries'][i]
    # range of rows where the answers to the MAS-IT questionnaire are located (from 126 to 170, step 2)
    for q_idx, n in enumerate(range(126, 171, 2)):
        try:
            answer = df_questionnaire.iloc[n, 1]
            if pd.notna(answer):
                # get just the number of the score, without the text of the answer
                mas_it_array[i, q_idx] = int(str(answer).split(' - ')[0])
            else:
                mas_it_array[i, q_idx] = np.nan
        except IndexError:
            print(IndexError)

# create the names of the columns for each mas_it question
question_cols = [f"mas_it_{j+1}" for j in range(23)]

# create a df with the mas_it answers
mas_it_df = pd.DataFrame(mas_it_array, columns=question_cols)

# insert a column for the participant_id
mas_it_df.insert(0, "participant_id", df_data_2024["participant_id"])

## Extraction of association and valence task

In [23]:
# funcition to build the df for each task taking as input the name of the task, the column numbers to extract and the names of the columns

def build_task_dataframe(task_name, col_nums, col_names):
    # created a dictionary of lists to store the association, it creates a default value for keys that don't exist yet
    results = defaultdict(list)

    for i in range(n_participants):
        # get the data of the desired task
        raw = df_raw_data_2024[task_name][i]

        # dynamically find start of cue words
        start_row = raw[raw.iloc[:, 0].str.contains("_cue", case=False, na=False)].index.min()
        # trim from after the instructions rows
        cleaned = raw.loc[start_row+2:].reset_index(drop=True)
        
        # loop through the rows of the dataframe (taking the maxium row number (shape[0]) of each particiants, so that the ones with fewer words can still get their words), step 2 
        for n in range(0, cleaned.shape[0], 2):
            # create the participant_id for each participant and cue word
            results["participant_id"].append(df_data_2024["participant_id"][i])
            # get the words looping through the rows and columns (excluding the participant_id col) of the df
            for j, col in zip(col_nums, col_names[1:]):
                try:
                    # lower and strip the words
                    val = cleaned.iloc[n, j]
                    if pd.notna(val) and str(val).strip().lower() not in ["nan", ""]:  # only apply lower/strip if not NaN
                        results[col].append(str(val).lower().strip())
                    else:
                        results[col].append(np.nan)
                except Exception:
                     # if row/col is missing, append NaN to keep lengths consistent
                    results[col].append(np.nan)

    # create a df with the values sorted by cue_word and participant_id
    df = pd.DataFrame(results).sort_values(by=['cue_word', 'participant_id']).reset_index(drop=True)

    # since some cue words were repeated for the same participant (e.g., 'psicologia' and 'mente' was duplicated for 4 participants), I drop the duplicates keeping the first one
    df = df.drop_duplicates(subset=['cue_word', 'participant_id'], keep='first').dropna(subset=['cue_word'])
  
    valid_ids = df.loc[df['cue_word'] == 'matematica', 'participant_id'].unique()
    df = df[df['participant_id'].isin(valid_ids)].reset_index(drop=True)

    # i also deleted some cue words that were inserted just for 1/2 participants (e.g., 'insegnamento' instead of 'insegnante')
    print(f"length of {task_name}_df: {len(df)} --> it should be n_participants * 40 = {len(valid_ids) * 40}\n")

    # check the number of cue words (each cue word should be repeated for 75 times, one for each participant)
    for word in df['cue_word'].unique():
        # we see that they have different length, some cue words are missing for some participants
        print(word, len(df[df['cue_word'] == word]))

    return df

In [ ]:
col_names_associations = ["participant_id", "cue_word", "association_1", "association_2", "association_3"]

associations_df = build_task_dataframe('associations', [0, 2, 4, 6], col_names_associations)

length of associations_df: 2797 --> it should be n_participants * 40 = 2800

ansia 70
attaccamento 70
benessere 70
biologia 70
cognizione 70
comportamento 70
creatività 70
curiosità 70
divertimento 70
emozione 70
equazione 70
esame 70
fisica 70
informatica 70
innovazione 70
insegnante 68
laurea 70
lavoro 70
matematica 70
media 70
mente 70
modello 70
neuroscienze 70
passione 70
personalità 70
professore 69
prova 70
psicologia 70
psicopatologia 70
psicoterapia 70
sapere 70
scienza 70
scienziato 70
sfida 70
statistica 70
sviluppo 70
terapeuta 70
test 70
università 70
valutazione 70


In [25]:
# names of the columns in the valence task
col_names_valence = ["participant_id", "cue_word", "valence_cue_word", "valence_association_1", "valence_association_2", "valence_association_3"]

valence_df = build_task_dataframe('valence', [0, 1, 3, 5, 7], col_names_valence)

length of valence_df: 2797 --> it should be n_participants * 40 = 2800

ansia 70
attaccamento 70
benessere 70
biologia 70
cognizione 70
comportamento 70
creatività 70
curiosità 70
divertimento 70
emozione 70
equazione 70
esame 70
fisica 70
informatica 70
innovazione 70
insegnante 68
laurea 70
lavoro 70
matematica 70
media 70
mente 70
modello 70
neuroscienze 70
passione 70
personalità 70
professore 69
prova 70
psicologia 70
psicopatologia 70
psicoterapia 70
sapere 70
scienza 70
scienziato 70
sfida 70
statistica 70
sviluppo 70
terapeuta 70
test 70
università 70
valutazione 70


### Merge data of associations and valence tasks values

In [26]:
cols_order = ['participant_id', 'cue_word', 'valence_cue_word', 'association_1', 'valence_association_1', 'association_2', 'valence_association_2', 'association_3', 'valence_association_3']

associations_valence = pd.merge(associations_df, valence_df, on = ['participant_id', 'cue_word'])
associations_valence = associations_valence[cols_order]

## Filtering of participants with  a lot missing data and .csv files creation 

In [ ]:
# associations_valence = associations_valence.replace("nan", np.nan)

missing_values = (
    associations_valence
    .set_index("participant_id")
    .isna()
    .mean(axis=1)
    .groupby(level=0)
    .mean() * 100                   
)
missing_values

# participant with a lot of missing data: 10, 36, 64 --> we removed these participants (Task_v13_120324.xlsx,Task_v7_160524.xlsx,Task_v22_160524.xlsx)

to_drop = missing_values[missing_values > 10].index
print("Participants to drop:", list(to_drop))

associations_valence_df = associations_valence[~associations_valence["participant_id"].isin(to_drop)]

final_participant_ids = associations_valence_df['participant_id'].unique()

associations_valence_df.to_csv(f'{path}/out/formatting/humans_2024/associations_valence_2024_humans.csv', index=False)

Participants to drop: ['student_exp1_010', 'student_exp1_036', 'student_exp1_064']


In [35]:
final_participant_ids

array(['student_exp1_001', 'student_exp1_002', 'student_exp1_003',
       'student_exp1_004', 'student_exp1_005', 'student_exp1_006',
       'student_exp1_007', 'student_exp1_008', 'student_exp1_009',
       'student_exp1_011', 'student_exp1_012', 'student_exp1_014',
       'student_exp1_015', 'student_exp1_016', 'student_exp1_017',
       'student_exp1_018', 'student_exp1_019', 'student_exp1_020',
       'student_exp1_021', 'student_exp1_022', 'student_exp1_023',
       'student_exp1_024', 'student_exp1_025', 'student_exp1_026',
       'student_exp1_027', 'student_exp1_028', 'student_exp1_029',
       'student_exp1_030', 'student_exp1_031', 'student_exp1_032',
       'student_exp1_033', 'student_exp1_034', 'student_exp1_035',
       'student_exp1_038', 'student_exp1_039', 'student_exp1_040',
       'student_exp1_041', 'student_exp1_043', 'student_exp1_044',
       'student_exp1_045', 'student_exp1_046', 'student_exp1_047',
       'student_exp1_048', 'student_exp1_049', 'student_exp1_0

In [ ]:
mas_it_df_filtered = mas_it_df[mas_it_df["participant_id"].isin(final_participant_ids)]
mas_it_df_filtered.to_csv(f'{path}/out/formatting/humans_2024/mas_it_2024_humans.csv', index=False)

len(mas_it_df_filtered['participant_id'].unique())

67